# Phase 5 — Construction des portefeuilles

## Pourquoi cette phase est importante

Les cours en CHF, l'univers convertible en CHF et le SARON sont déjà publiés. Cette phase en fait trois portefeuilles. Les trois utilisent les mêmes titres et les mêmes rendements mensuels. Seule la règle de poids change. On peut donc attribuer l'écart de résultat à la règle, et à elle seule.

### Portefeuille 1 — Équipondéré

Objectif : comprendre l'effet de la diversification simple.

Chaque titre retenu reçoit le même poids. Le rendement passé et le risque passé ne servent pas à choisir ce poids.

### Portefeuille 2 — Volatilité minimale

Objectif : comprendre les mécanismes de réduction du risque.

Les poids sont ceux qui rendent la variance du portefeuille la plus petite. Un titre calme, ou un titre qui ne baisse pas en même temps que les autres, reçoit davantage. Le calcul s'appuie sur la covariance des rendements mensuels en CHF.

### Portefeuille 3 — Sharpe maximal

Objectif : comprendre la notion de rendement ajusté au risque.

Les poids maximisent le rendement au-dessus du SARON, par unité de risque. Le portefeuille peut accepter une volatilité plus haute lorsque le rendement supplémentaire est plus que proportionnel.

## Les trois livrables

| Livrable | Fichier | Rôle |
|---|---|---|
| Documentation des stratégies | `documentation/05_portfolio_construction.ipynb` | La méthode, les formules, les fichiers produits |
| Calculs détaillés | `reports/portfolio/05_calculs_detailles.ipynb` | Le même calcul repris sur un mois, puis sur tous les mois de la période |
| Analyse comparative | `notebooks/05_analyse_comparative.ipynb` | La lecture des écarts entre les trois portefeuilles |

La vérification automatique est à part, dans `tests/portfolio/phase_05_validation.ipynb`. Elle ne remplace aucun des trois livrables.

## Sources du registre

Cette phase ne télécharge rien. Elle n'ouvre pas non plus le registre. Elle lit trois fichiers déjà préparés. Le tableau permet de remonter de ces fichiers jusqu'au jeu d'origine.

| Fichier lu | Jeu du registre | Ce que le jeu apporte |
|---|---|---|
| `data/02_processed/market/market_prices_chf.csv` | `DS-003`, cours Yahoo | Le cours de chaque titre |
| le même fichier | `DS-004`, taux de change BNS | La conversion en CHF du cours qui n'est pas déjà en francs |
| `data/03_universe/eligible/chf_portfolio_universe.csv` | `DS-002`, master des titres | Le titre, la place, la devise de cotation |
| le même fichier | `DS-004` | Une devise sans série BNS est hors univers |
| `data/02_processed/macro/saron.csv` | `DS-005`, SARON | Le taux sans risque, un mois à la fois |

En remontant encore, les cours viennent de cette chaîne : `DS-000` positions du benchmark, `DS-001` candidats, `DS-002` master, `DS-003` cours. Le change `DS-004` ne demande que les devises présentes dans le master. Le SARON `DS-005` n'a pas de jeu en amont.

Le cours en CHF contient déjà l'échelle de cotation, par exemple les pence de Londres. Cette phase n'ouvre donc pas `data/02_processed/market/quote_currency.csv`.

Le registre d'éligibilité, `DS-006`, est constitué. Il ne sert pas encore à construire les portefeuilles : chaque statut est Indeterminate, donc aucun titre n'est écarté par cette lecture.


## Entrées

Le module lit trois fichiers déjà traités. Il ne réécrit pas `data/01_raw`.

| Rôle | Fichier | Colonnes utilisées |
|---|---|---|
| Cours en CHF | `data/02_processed/market/market_prices_chf.csv` | `symbol`, `timestamp`, `close_chf` |
| Univers | `data/03_universe/eligible/chf_portfolio_universe.csv` | `yahoo_symbol`, `ticker`, `name`, `exchange`, `listing_currency`, `chf_status`, `exclusion_reason` |
| Taux sans risque | `data/02_processed/macro/saron.csv` | `series_id`, `date`, `value`, `rate_currency` |

`close_chf` est le cours déjà converti en francs, échelle de cotation comprise. Le SARON est le taux au jour le jour de la BNS, en pourcentage annuel, une observation par mois. Sa devise dans le fichier est le CHF.

Le statut ODD est Indeterminate sur tout le master. Aucune ligne n'est Eligible. Il n'entre pas dans le choix des titres. Un titre retenu est un titre déjà convertible en CHF, avec un cours de fin de mois strictement positif sur les soixante mois qui précèdent la période et sur toute la période.

## Période commune

La longueur vient de `MARKET_HISTORY_YEARS` dans `src/config.py`. La valeur du projet est 10 ans. La fin est le dernier mois du SARON. Avec une fin $T$ et une longueur de $Y$ années, le premier mois de rendement est

$$
t_1 = T - 12Y + 1
$$

Pour calculer le rendement de $t_1$, il faut le cours de fin du mois précédent. Pour fixer le poids de $t_1$, il faut aussi les soixante rendements qui précèdent $t_1$. Les cours nécessaires commencent donc soixante et un mois avant $t_1$. Le nombre de mois de rendement publié reste $12Y$. La cellule en bas de ce notebook affiche $T$, $t_1$ et le nombre de fins de mois exigées. Un mois de SARON plus récent décale toute la période. Des cours plus récents que $T$ attendent le SARON de ce mois.

Un mois sans SARON dans cette période arrête le calcul. Le taux n'est pas interpolé.

## Qui entre dans les trois portefeuilles

Le cours de fin de mois du titre $i$ est le dernier `close_chf` observé pendant le mois $m$ :

$$
P_{i,m} = \text{dernier close CHF de } i \text{ dans le mois } m
$$

Un titre entre dans les trois portefeuilles lorsqu'il remplit toutes les conditions suivantes.

1. `chf_status` vaut `included` dans l'univers CHF. Sinon le motif reste celui de la phase 4, `no_snb_series`.
2. Il a au moins un cours dans l'historique. Sinon le motif est `no_price_history`.
3. Aucun cours de la période n'est négatif ou nul. Sinon le motif est `non_positive_close`.
4. Toutes les fins de mois exigées, de $t_1 - 1$ à $T$, sont présentes et strictement positives. Sinon le motif est `incomplete_monthly_history`.
5. La série de rendements n'est pas constante. Sinon le motif est `constant_monthly_return`.

Ces décisions sont écrites dans `portfolio_membership.csv`. Les trois stratégies reçoivent ensuite exactement la même liste, dans `decision = included`. Un titre écarté n'a aucun poids.

## Rendement mensuel d'un titre

Le rendement simple du mois $t$ utilise deux cours de fin de mois consécutifs :

$$
R_{i,t} = \frac{P_{i,t}}{P_{i,t-1}} - 1
$$

$t$ et $t-1$ sont deux mois civils qui se suivent. Si l'un des deux cours manque, le rendement de ce mois n'est pas calculé et le titre sort de la liste. Aucun cours n'est remplacé par le précédent.

Le rendement est déjà en CHF, parce que $P$ est `close_chf`. Le change du mois est donc dans le rendement du mois où le taux SNB change.

## Taux sans risque

Le fichier SARON donne un pourcentage annuel. Le Sharpe mensuel a besoin d'un rendement sur le mois. Pour le mois $t$ :

$$
r_{f,t} = \left(1 + \frac{\mathrm{SARON}_t}{100}\right)^{1/12} - 1
$$

$\mathrm{SARON}_t$ est le taux du mois pendant lequel le rendement du portefeuille est gagné. Le rendement mesuré entre la fin du mois $t-1$ et la fin du mois $t$ retire le SARON du mois $t$.

Le facteur 252 du calendrier quotidien ne s'applique pas ici. La série commune est mensuelle, comme le SARON et comme le taux de change SNB. L'annualisation des métriques utilise donc 12.

L'excédent du portefeuille est

$$
E_{p,t} = R_{p,t} - r_{f,t}
$$

## Rendement du portefeuille

Le poids d'un trimestre est fixé avec les 60 mois de rendements qui précèdent son premier mois, puis tenu pendant les trois mois. Aucun émetteur ne dépasse 5 %, aucun secteur 20 %, aucune zone 50 % et aucune devise de cotation 40 %. Chaque mois, le rendement du portefeuille est

$$
R_{p,t} = \sum_{i=1}^{N} w_i \, R_{i,t}
$$

C'est le rendement d'un portefeuille ramené à ces poids au début de chaque mois. Aucun coût de transaction n'est retiré.

Contraintes, pour les trois stratégies :

$$
w_i \ge 0, \qquad \sum_{i=1}^{N} w_i = 1
$$

Le portefeuille est donc entièrement investi, sans vente à découvert et sans levier. Après le solveur, un poids négatif dû seulement à l'arrondi est ramené à zéro, puis les poids sont redivisés par leur somme.

## Portefeuille 1 — Équipondéré

$$
w_i = \frac{1}{N}
$$

$N$ est le nombre de titres retenus. Le poids ne dépend pas de $R_{i,t}$.

Ce portefeuille sert de base. Le point de départ est $1/N$, ramené ensuite aux plafonds de secteur, de zone et de devise. Toute la liste reste investie. La bande de 25 à 40 lignes s'applique aux deux livres optimisés, pas à celui-ci : retirer des titres serait une règle de sélection que l'IPS ne donne pas.

## Portefeuille 2 — Volatilité minimale

À chaque début de trimestre, la covariance est estimée sur les 60 mois qui précèdent ce mois. C'est l'estimateur de Ledoit et Wolf (2004) : la covariance d'échantillon, diviseur $T$, rétrécie vers la variance moyenne. L'intensité est calculée sur ces 60 mois et bornée entre 0 et 1.

Le programme cherche

$$
\min_w \; w^\top \Sigma w
$$

sous les contraintes $w_i \ge 0$, $w_i \le 0{,}05$, un secteur au plus à 20 %, une zone au plus à 50 %, une devise de cotation au plus à 40 %, et $\sum_i w_i = 1$. Les deux livres optimisés sont ensuite ramenés entre 25 et 40 lignes. Si la rotation brute dépasse 12,5 % par rapport au trimestre précédent, le livre est rapproché de celui-ci, de façon à rester sous 50 % sur l'année.

$w^\top \Sigma w$ est la variance mensuelle du rendement du portefeuille sur cet échantillon. Le solveur est SLSQP. Le point de départ est l'équipondéré, qui est un portefeuille autorisé : sur ces 60 mois, le minimum obtenu ne peut pas avoir une variance plus haute que lui. Cette comparaison ne porte pas sur la série publiée.

La réduction du risque, sur l'échantillon, passe par la covariance : un titre peut recevoir un poids faible même s'il est peu volatile, lorsqu'il bouge comme le reste du portefeuille. Elle passe aussi par la concentration, dans la limite de 5 % par émetteur et des plafonds de secteur, de zone et de devise. La volatilité annualisée publiée est l'écart-type de la série $R_{p,t}$, pas cette variance d'échantillon.


## Portefeuille 3 — Sharpe maximal

Le Sharpe d'un vecteur de poids, sur les 60 mois qui précèdent le mois noté, est

$$
S(w) = \frac{w^\top \mu}{\sqrt{w^\top \Sigma_e w}}
$$

$\mu$ est la moyenne des rendements excédentaires de ces 60 mois. $\Sigma_e$ est leur covariance de Ledoit-Wolf. Retirer un taux qui change au cours du temps modifie les variances : $\Sigma_e$ n'est pas la covariance des rendements bruts.

Les plafonds empêchent de traiter le problème comme un vecteur libre, puis de le ramener ensuite à une somme de 1. Le programme maximise donc $S(w)$ directement, avec $0 \le w_i \le 0{,}05$, les plafonds de groupe, et $\sum_i w_i = 1$.

SLSQP est lancé depuis l'équipondéré, depuis le titre dont l'excédent moyen est le plus haut (ce titre reçoit 5 %, le reste est réparti à parts égales), puis depuis huit autres points de départ tirés avec la graine fixe 15. Le vecteur conservé est celui dont le Sharpe est le plus haut sur ces 60 mois. Il doit au moins atteindre le Sharpe de l'équipondéré sur le même échantillon.

La comparaison à l'équipondéré se fait sur l'échantillon qui sert à choisir le poids, pas sur la série publiée. Le rendement, la volatilité et le ratio de Sharpe publiés sont ceux du chemin obtenu en enchaînant ces décisions. L'équipondéré ne lit pas les rendements : son poids est la projection de $1/N$ sur les plafonds, et il est le même chaque trimestre.


## Métriques publiées

Toutes les métriques sont calculées sur les $T$ mois de la période.

Rendement annualisé, géométrique :

$$
R_p^{\mathrm{ann}} = \left(\prod_{t=1}^{T} (1 + R_{p,t})\right)^{12/T} - 1
$$

Volatilité annualisée, écart-type d'échantillon des rendements du portefeuille :

$$
\sigma_p^{\mathrm{ann}} = \sqrt{ \frac{1}{T-1} \sum_{t=1}^{T} \left(R_{p,t} - \bar R_p\right)^2 } \times \sqrt{12}
$$

Sharpe annualisé. La moyenne et l'écart-type portent sur l'excédent $E_{p,t}$, pas sur $R_{p,t}$ :

$$
\mathrm{Sharpe}_p = \frac{\bar E_p}{s_E} \times \sqrt{12}
$$

$s_E$ est l'écart-type d'échantillon de $E_{p,t}$.

Drawdown maximal. La performance cumulée part de 1 avant le premier mois :

$$
V_t = \prod_{k=1}^{t} (1 + R_{p,k})
$$

$$
\mathrm{DD}_t = \frac{V_t}{\max_{k \le t} V_k} - 1
\qquad
\mathrm{MDD} = \min_t \mathrm{DD}_t
$$

Le nombre publié est négatif. Une valeur $-0{,}17$ décrit une perte de 17 % depuis un sommet.

Nombre effectif de titres :

$$
N_{\mathrm{eff}} = \frac{1}{\sum_{i=1}^{N} w_i^2}
$$

Pour l'équipondéré, $N_{\mathrm{eff}} = N$. Plus les poids se concentrent, plus $N_{\mathrm{eff}}$ baisse. Un seul titre donne $N_{\mathrm{eff}} = 1$.

## Création des CSV

Le code est `src/portfolio_construction/build_portfolios.py`. Depuis la racine du projet :

```text
python -m src.pipeline.refresh
```

Cette commande acquiert les jeux de données dont la fréquence du registre est due, remplace leur fichier brut, recharge la base, complète les devises de cotation manquantes, retraite les cours en CHF, reconstruit les trois portefeuilles, le suivi et la performance, puis relance le contrôle qualité. Le détail de l'ordre et des fréquences est dans `documentation/03_data_acquisition.ipynb`.

Pour reconstruire seulement les cinq CSV, une fois les fichiers traités à jour :

```text
python -m src.portfolio_construction.build_portfolios
```

La fonction `build_portfolios` enchaîne les étapes suivantes.

1. Lire les cours CHF, l'univers et le SARON.
2. Garder le dernier cours de chaque titre et de chaque mois.
3. Fixer la période : $12 \times$ `MARKET_HISTORY_YEARS` mois, terminée au dernier mois du SARON.
4. Écrire la décision de chaque titre du master.
5. Calculer le panneau de rendements des titres retenus.
6. Calculer les trois vecteurs de poids.
7. Calculer, pour chaque vecteur, la série mensuelle et les métriques.
8. Écrire les cinq CSV.

`data/03_universe/` reste la liste des titres : candidat, éligible, exclu. Les poids, les rendements et les métriques sont une couche suivante. Ils sont écrits dans `data/04_portfolios/`. Chaque CSV est d'abord écrit dans un fichier temporaire du même dossier, puis ce fichier remplace la version précédente. Le notebook qui refait l'arithmétique est dans `reports/portfolio/`.

### `portfolio_membership.csv`

Une ligne par titre du master.

| Colonne | Sens |
|---|---|
| `yahoo_symbol`, `ticker`, `name`, `exchange`, `listing_currency` | Identité du titre |
| `decision` | `included` ou `excluded` |
| `reason` | Motif d'écart, vide lorsque le titre est retenu |
| `available_price_months` | Fins de mois strictement positives trouvées dans la période |
| `required_price_months` | Fins de mois exigées, période publiée et soixante mois précédents |

### `portfolio_weights.csv`

Une ligne par stratégie et par titre retenu. Un poids nul est conservé : il montre que le solveur a pu écarter un titre.

| Colonne | Sens |
|---|---|
| `strategy` | `equal_weight`, `minimum_volatility` ou `maximum_sharpe` |
| `yahoo_symbol`, `ticker`, `name`, `listing_currency` | Identité du titre |
| `weight` | Part du portefeuille, entre 0 et 1 |

### `portfolio_monthly_returns.csv`

Une ligne par mois et par stratégie. C'est le détail du calcul.

| Colonne | Sens |
|---|---|
| `month` | Mois du rendement, `YYYY-MM` |
| `strategy` | L'une des trois stratégies |
| `portfolio_return` | $R_{p,t}$ |
| `risk_free_return` | $r_{f,t}$, rendement mensuel du SARON |
| `excess_return` | $R_{p,t} - r_{f,t}$ |

### `portfolio_metrics.csv`

Une ligne par stratégie.

| Colonne | Sens |
|---|---|
| `strategy` | L'une des trois stratégies |
| `names` | Nombre de titres du vecteur de poids |
| `months` | Nombre de mois de rendement, soit $12Y$ |
| `window_start`, `window_end` | Premier et dernier mois de rendement |
| `annualized_return` | $R_p^{\mathrm{ann}}$ |
| `annualized_volatility` | $\sigma_p^{\mathrm{ann}}$, volatilité de $R_{p,t}$ |
| `sharpe_ratio` | Sharpe annualisé, excédent au-dessus du SARON |
| `maximum_drawdown` | Drawdown maximal, nombre négatif |
| `maximum_weight` | Plus grand poids |
| `effective_holdings` | $N_{\mathrm{eff}}$ |
| `risk_free_series` | `SARON` |
| `risk_free_currency` | Devise de reporting du projet |

### `portfolio_comparison.csv`

Une ligne par métrique. Les deux dernières colonnes sont l'écart avec l'équipondéré : valeur de la stratégie moins valeur de l'équipondéré. Pour la volatilité, un écart négatif est une volatilité plus basse. Pour le drawdown maximal, un écart positif rapproche le nombre de zéro, donc un repli moins profond.

| Colonne | Sens |
|---|---|
| `metric` | Nom de la métrique, le même que dans `portfolio_metrics.csv` |
| `equal_weight` | Valeur de l'équipondéré |
| `minimum_volatility` | Valeur de la volatilité minimale |
| `maximum_sharpe` | Valeur du Sharpe maximal |
| `minimum_volatility_minus_equal_weight` | Écart de la volatilité minimale |
| `maximum_sharpe_minus_equal_weight` | Écart du Sharpe maximal |

La cellule suivante lit ces fichiers et en montre la forme. Elle ne relance pas le calcul.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import MARKET_HISTORY_YEARS, PORTFOLIO_DATA, REPORTING_CURRENCY

print(f"Devise de reporting : {REPORTING_CURRENCY}")
print(f"Durée de la période, en années : {MARKET_HISTORY_YEARS}")
print()

for path in (
    PORTFOLIO_DATA / "portfolio_membership.csv",
    PORTFOLIO_DATA / "portfolio_weights.csv",
    PORTFOLIO_DATA / "portfolio_monthly_returns.csv",
    PORTFOLIO_DATA / "portfolio_metrics.csv",
    PORTFOLIO_DATA / "portfolio_comparison.csv",
):
    frame = pd.read_csv(path)
    print(f"{path.relative_to(PROJECT_ROOT)} : {len(frame)} lignes, colonnes = {', '.join(frame.columns)}")

membership = pd.read_csv(PORTFOLIO_DATA / "portfolio_membership.csv", dtype=str, keep_default_na=False)
print()
print(membership.groupby(["decision", "reason"], dropna=False).size().rename("titres").to_string())

metrics = pd.read_csv(PORTFOLIO_DATA / "portfolio_metrics.csv")
window = metrics.loc[:, ["strategy", "names", "months", "window_start", "window_end"]]
print()
print("Période de cette exécution")
print(window.to_string(index=False))
required = sorted(membership["required_price_months"].unique())
print(f"Fins de mois exigées : {', '.join(required)}")


Devise de reporting : CHF
Durée de la période, en années : 10

data\04_portfolios\portfolio_membership.csv : 120 lignes, colonnes = yahoo_symbol, ticker, name, exchange, listing_currency, decision, reason, available_price_months, required_price_months
data\04_portfolios\portfolio_weights.csv : 26640 lignes, colonnes = strategy, month, yahoo_symbol, ticker, name, listing_currency, weight
data\04_portfolios\portfolio_monthly_returns.csv : 360 lignes, colonnes = month, strategy, portfolio_return, risk_free_return, excess_return
data\04_portfolios\portfolio_metrics.csv : 3 lignes, colonnes = strategy, names, months, window_start, window_end, annualized_return, annualized_volatility, sharpe_ratio, maximum_drawdown, maximum_weight, effective_holdings, risk_free_series, risk_free_currency
data\04_portfolios\portfolio_comparison.csv : 6 lignes, colonnes = metric, equal_weight, minimum_volatility, maximum_sharpe, minimum_volatility_minus_equal_weight, maximum_sharpe_minus_equal_weight

decision

## Suite de la lecture

Le détail arithmétique, du cours de fin de mois jusqu'au Sharpe, est dans `reports/portfolio/05_calculs_detailles.ipynb`.

La lecture des écarts entre les trois règles de poids est dans `notebooks/05_analyse_comparative.ipynb`.

Le suivi et la performance liront `portfolio_weights.csv` et `portfolio_monthly_returns.csv`. Le fichier de poids a une ligne par stratégie, par mois et par titre : c'est le poids qui a été tenu ce mois-là.

# Dictionnaire des termes de cette note

Ce dictionnaire reprend les termes employés dans cette note. Il n'en ajoute pas. Les formules restent dans la section qui les écrit.

**Annualisation.** Passage d'une série mensuelle à un chiffre d'année, avec le facteur 12.

**Banque nationale suisse (BNS, SNB).** Source du change et du SARON déjà intégrés dans les fichiers lus.

**Benchmark.** Fichier de positions du proxy investissable, `DS-000`. La référence stratégique est l'indice MSCI ACWI Sustainable Development Index (Net) : elle n'a pas de cours propre et n'est pas ce fichier.

**Close CHF (`close_chf`).** Dernier cours du mois déjà converti en francs, échelle de cotation comprise.

**Covariance.** Tableau de la façon dont les titres varient ensemble. Elle est estimée par Ledoit et Wolf (2004) sur 60 mois.

**Devise de cotation.** Monnaie du cours en bourse. Plafond de 40 % du portefeuille.

**Drawdown maximal.** Plus forte baisse de la performance cumulée depuis un plus haut déjà atteint.

**Écart-type.** Mesure de dispersion. L'écart-type d'échantillon utilise le diviseur T − 1.

**Émetteur.** Société du titre. Aucun émetteur ne dépasse 5 %.

**Eligible.** Statut ODD qui autoriserait l'entrée. Aucune ligne ne l'a dans cette note.

**Entièrement investi.** La somme des poids vaut 1.

**Équipondéré.** Point de départ 1/N, puis ramené aux plafonds. Toute la liste reste investie. La bande de 25 à 40 lignes ne s'applique pas à ce livre.

**Excédent.** Rendement du portefeuille moins le taux sans risque du mois.

**Indeterminate.** Statut ODD de l'évaluation `DS-006`. Il n'écarte aucun titre.

**IPS.** Politique d'investissement. Elle ne dit pas quels titres retirer de la liste complète de l'équipondéré.

**Jeu du registre.** Ensemble de données identifié par un code `DS-`.

**Ledoit-Wolf (2004).** Estimateur qui rétrécit la covariance d'échantillon vers la variance moyenne.

**Levier.** Investir plus que le capital. Le portefeuille n'en a pas : les poids sont positifs ou nuls et somment à 1.

**Projection.** Ramener le vecteur 1/N sous les plafonds.

**Rendement.** Variation entre deux cours de fin de mois consécutifs, déjà en francs.

**Rotation brute.** Somme des changements de poids d'un trimestre à l'autre. Plafond de 12,5 % par rééquilibrage.

**SARON.** Taux au jour le jour du franc, en pourcentage annuel dans le fichier. Taux sans risque du mois. Un mois sans SARON arrête le calcul.

**Secteur.** Groupe de métier plafonné à 20 %.

**Sharpe.** Rendement excédentaire divisé par son écart-type. Le livre de Sharpe maximal maximise ce ratio sur les 60 mois déjà connus.

**Sharpe maximal.** Portefeuille qui maximise le Sharpe sur l'échantillon de 60 mois, sous les plafonds.

**SLSQP.** Solveur qui cherche les poids de la volatilité minimale et du Sharpe maximal.

**Taux sans risque.** SARON du mois, converti en rendement mensuel.

**Variance.** Carré de l'ampleur des variations. Le livre de volatilité minimale minimise la variance du portefeuille sur l'échantillon.

**Vente à découvert.** Poids négatif. Elle est exclue par la contrainte de poids positifs ou nuls.

**Volatilité.** Écart-type des rendements, annualisé avec la racine de 12 sur la série publiée.

**Volatilité minimale.** Portefeuille dont la variance d'échantillon est la plus faible sous les plafonds.

**Yahoo.** Source d'origine des cours, jeu `DS-003`, déjà convertis en francs dans le fichier lu.

**Zone.** Groupe de pays plafonné à 50 %.
